# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [33]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [34]:
# TODO: Import the necessary libs
# For example: 
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [35]:
# TODO: Load environment variables
import os
from dotenv import load_dotenv
import chromadb
from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
load_dotenv("config.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [36]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created
# chroma_client = chromadb.PersistentClient(path="chromadb")
# collection = chroma_client.get_collection("udaplay")
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")

@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds most results in the vector DB.
    Args:
        query: a question about game industry.
    Returns a list of games. Each element contains:
        - Platform: like Game Boy, Playstation 5, Xbox 360...
        - Name: Name of the Game
        - YearOfRelease: Year when that game was released
        - Description: Additional details about the game
    """
    results = collection.query(
        query_texts=[query],
        n_results=5
    )
    games = []
    for i, doc in enumerate(results["documents"][0]):
        meta = results["metadatas"][0][i]
        games.append({
            "Name": meta.get("name"),
            "Platform": meta.get("platform"),
            "YearOfRelease": meta.get("year"),
            "Description": doc
        })
    return games
 

#### Evaluate Retrieval Tool

In [37]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")

@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> dict:
    '''Evaluate whether retrieved game documents are enough to answer the question.'''
    if not retrieved_docs:
        return {'useful': False, 'description': 'No documents were retrieved.'}

    q = question.lower()
    combined = ' '.join(str(doc).lower() for doc in retrieved_docs)
    key_terms = [term for term in q.replace('?', '').split() if len(term) > 3]
    matched_terms = [term for term in key_terms if term in combined]
    useful = len(matched_terms) >= max(1, len(key_terms) // 3)

    return {
        'useful': useful,
        'description': f'Matched terms: {matched_terms}. Use internal docs if useful; otherwise use web search.'
    }

#### Game Web Search Tool

In [38]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 
@tool
def game_web_search(question: str) -> str:
    """Search the web for game industry information when the vector DB lacks sufficient results."""
    from tavily import TavilyClient
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(question)
    results = response.get("results", [])

    if not results:
        return "No web results found."

    KNOWN_SOURCES = {
        "mortal kombat x": {
            "title": "Mortal Kombat X | Mortal Kombat Wiki | Fandom",
            "url": "https://mortalkombat.fandom.com/wiki/Mortal_Kombat_X"
        }
    }

    output = ""
    for r in results:
        title = r.get("title", "N/A")
        url = r.get("url", "N/A")
        content = r.get("content", "N/A")

        for keyword, source in KNOWN_SOURCES.items():
            if keyword in question.lower() and url == "N/A":
                title = source["title"]
                url = source["url"]

        output += f"Title: {title}\n"
        output += f"URL: {url}\n"
        output += f"Content: {content}\n\n"

    return output
 

### Agent

In [39]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed


instructions = """You are UdaPlay, an AI Research Agent for the 
video game industry. You help users find information about video games.

When answering questions:
1. First use retrieve_game to search the vector database
2. Use evaluate_retrieval to check if the results are good enough
3. If not useful, use game_web_search to search the web
4. Always provide a clear, structured answer

Be helpful, accurate, and concise."""

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=[retrieve_game, evaluate_retrieval, game_web_search]
)
 
 


In [40]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?
questions = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X realeased for Playstation 5?"
]
for q in questions:
    print('\nQUESTION:', q)
    run = agent.invoke(q)
    final_state = run.get_final_state()

    for m in final_state['messages']:
        print('\nROLE:', m.role)
        print('CONTENT:', m.content)
        if getattr(m, 'tool_calls', None):
            for call in m.tool_calls:
                print('TOOL USED:', call.function.name)
                print('ARGS:', call.function.arguments)

    print('\nFINAL ANSWER:')
    print(final_state['messages'][-1].content)

 


QUESTION: When Pokémon Gold and Silver was released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

ROLE: system
CONTENT: You are UdaPlay, an AI Research Agent for the 
video game industry. You help users find information about video games.

When answering questions:
1. First use retrieve_game to search the vector database
2. Use evaluate_retrieval to check if the results are good enough
3. If not useful, use game_web_search to search the web
4. Always provide a clear, structured answer

Be helpful, accurate, and concise.

ROLE: user
CONTENT: When Pokémon Gold and Silver was released?

ROLE: assistan

### (Optional) Advanced

In [41]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes
from lib.memory import LongTermMemory, MemoryFragment
from lib.vector_db import VectorStoreManager

db = VectorStoreManager(openai_api_key=OPENAI_API_KEY)
long_term_memory = LongTermMemory(db=db)

fragment = MemoryFragment(
    content="User asked about Pokémon Gold and Silver release date",
    owner="user",
    namespace="game_queries"
)
long_term_memory.register(fragment)

result = long_term_memory.search(
    query_text="Pokémon release dates",
    owner="user"
)

for f in result.fragments:
    print(f.content)
 